# Coral Classifier Training
Download a Roboflow object-detection dataset, crop each bounding box into a per-class image, and train a YOLOv8 classification model for use in **coralX**.

**Steps:**
1. Fill in your config in the *Configuration* cell
2. Runtime → Change runtime type → **T4 GPU**
3. Runtime → Run all
4. Download `best.pt` from the last cell and load it in coralX → AI Auto-Label

In [ ]:
# @title Install dependencies
!pip install -q roboflow ultralytics opencv-python pyyaml albumentations

In [ ]:
# @title Configuration { display-mode: "form" }

# -- Roboflow --
API_KEY   = ""  # @param {type:"string"}
WORKSPACE = ""  # @param {type:"string"}
PROJECT   = ""  # @param {type:"string"}
VERSION   = 1   # @param {type:"integer"}

# -- Training --
EPOCHS     = 100  # @param {type:"integer"}
IMGSZ      = 64   # @param {type:"integer"}
BATCH      = 32   # @param {type:"integer"}
BASE_MODEL = "yolov8n-cls.pt"  # @param ["yolov8n-cls.pt", "yolov8s-cls.pt", "yolov8m-cls.pt"]

# -- Augmentation (applied during training) --
AUG_FLIPLR  = 0.5   # @param {type:"number"}  horizontal flip probability
AUG_FLIPUD  = 0.3   # @param {type:"number"}  vertical flip probability
AUG_DEGREES = 15.0  # @param {type:"number"}  rotation ± degrees
AUG_HSV_H   = 0.015 # @param {type:"number"}  hue shift
AUG_HSV_S   = 0.4   # @param {type:"number"}  saturation shift
AUG_HSV_V   = 0.3   # @param {type:"number"}  brightness shift
AUG_BLUR    = 0.1   # @param {type:"number"}  blur probability (simulates turbid water)

# -- Output --
SAVE_TO_DRIVE = True           # @param {type:"boolean"}
DRIVE_FOLDER  = "coralX_models"  # @param {type:"string"}

In [ ]:
# @title Mount Google Drive (skip if SAVE_TO_DRIVE is False)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

In [ ]:
# @title 1. Download dataset from Roboflow
from roboflow import Roboflow

rf = Roboflow(api_key=API_KEY)
dataset = (
    rf.workspace(WORKSPACE)
    .project(PROJECT)
    .version(VERSION)
    .download("yolov8", location="/content/dataset/raw")
)
DATASET_DIR = dataset.location
print(f"Dataset downloaded to: {DATASET_DIR}")

In [ ]:
# @title 2. Crop bounding boxes → per-class folders
import cv2
import yaml
from pathlib import Path


def bbox_to_pixels(
    cx: float, cy: float, bw: float, bh: float, img_w: int, img_h: int
) -> tuple:
    x1 = max(0, int((cx - bw / 2) * img_w))
    y1 = max(0, int((cy - bh / 2) * img_h))
    x2 = min(img_w, int((cx + bw / 2) * img_w))
    y2 = min(img_h, int((cy + bh / 2) * img_h))
    return x1, y1, x2, y2


def find_image(images_dir: Path, stem: str) -> Path | None:
    for ext in (".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"):
        p = images_dir / (stem + ext)
        if p.exists():
            return p
    return None


def crop_split(images_dir: Path, labels_dir: Path, out_dir: Path, class_names: list) -> int:
    count = 0
    for label_file in sorted(labels_dir.glob("*.txt")):
        img_path = find_image(images_dir, label_file.stem)
        if img_path is None:
            continue
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img_h, img_w = img.shape[:2]
        for i, line in enumerate(label_file.read_text().strip().splitlines()):
            parts = line.split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            x1, y1, x2, y2 = bbox_to_pixels(
                float(parts[1]), float(parts[2]),
                float(parts[3]), float(parts[4]),
                img_w, img_h,
            )
            crop = img[y1:y2, x1:x2]
            if crop.size == 0:
                continue
            cls_name = class_names[cls_id] if cls_id < len(class_names) else str(cls_id)
            dest = out_dir / cls_name
            dest.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(dest / f"{label_file.stem}_{i}.jpg"), crop)
            count += 1
    return count


dataset_dir = Path(DATASET_DIR)
cls_dir = Path("/content/dataset/classify")
class_names = yaml.safe_load((dataset_dir / "data.yaml").read_text())["names"]
print(f"Classes: {class_names}")

for split in ("train", "valid", "test"):
    images_dir = dataset_dir / split / "images"
    labels_dir = dataset_dir / split / "labels"
    if not images_dir.exists():
        print(f"  [skip] {split} not present")
        continue
    n = crop_split(images_dir, labels_dir, cls_dir / split, class_names)
    print(f"  {split}: {n} crops saved")

In [ ]:
# @title 3. Train YOLOv8 classification with augmentation
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
model.train(
    task="classify",
    data=str(cls_dir),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    name="coral_classifier",
    exist_ok=True,
    # Augmentation — tuned for underwater coral imagery
    fliplr=AUG_FLIPLR,
    flipud=AUG_FLIPUD,
    degrees=AUG_DEGREES,
    hsv_h=AUG_HSV_H,
    hsv_s=AUG_HSV_S,
    hsv_v=AUG_HSV_V,
    blur=AUG_BLUR,
)

In [ ]:
# @title 4. Save / download best.pt
import shutil
from pathlib import Path

best = Path("runs/classify/coral_classifier/weights/best.pt")

if not best.exists():
    print("ERROR: best.pt not found — check training output above.")
else:
    if SAVE_TO_DRIVE:
        drive_dir = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}")
        drive_dir.mkdir(parents=True, exist_ok=True)
        dest = drive_dir / "best.pt"
        shutil.copy(best, dest)
        print(f"Saved to Google Drive: {dest}")

    from google.colab import files
    print("Downloading best.pt to your computer…")
    files.download(str(best))